# 07 Separator (IA-SEP-001)

**Requirement:** `IA-SEP-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

## Governing equations


\(R_\mathrm{ionic} = L\,\tau / (\sigma A \varepsilon s^{1.5})\). Separator blocks dissolved O2
and bubbles from iron without a large ionic penalty, remaining hydrophilic/macroporous
(US FIG. 4A; EP Claim 2, [0048]). Wrapped stacked-core pores 1 µm to 1 cm (US FIG. 36A).


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path
setup_path()
from plant_sim.params import default_params, CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G
from plant_sim.plotting import apply_style, timeseries, xy_plot, FIGURES
apply_style()
P = default_params()
print("6 M KOH default", P.c_KOH_mol_m3/1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G,2), round(CAPACITY_MAGNETITE_MAH_G,2))
print("figures ->", FIGURES)


## Isolated component simulation

In [ ]:
from plant_sim.components.separator import Separator
from plant_sim.components.base import integrate_component
sep = Separator(P)
t, y = integrate_component(lambda t,y: sep.rhs(t,y,{"T_ely_K": P.T_ep_sim_K, "c_KOH_mol_m3": 6000, "I_cell_A": 5.0, "s_wet": 0.95, "c_O2_ely_mol_m3": 0.02}, {}), sep.y0(), (0, 800), n_eval=100)
R = [sep.outputs(float(ti), y[:,i], {"c_KOH_mol_m3": 6000, "I_cell_A": 5.0}, {})["R_ionic_ohm"] for i,ti in enumerate(t)]
timeseries(t, {"T": y[0], "s": y[1], "c_O2": y[2], "R": R}, xlabel="t (s)", ylabel="mixed", title="Separator R_ionic and O2", callout="IA-SEP-001", stem="nb07_sep")


## Verification against patents / SSS002

In [ ]:
print('R_ionic is algebraic in conductivity, thickness, area, saturation.')